# 001_chunking - Text Chunking Strategies
Cuaderno práctico basado en la lección **Text chunking strategies** (*Building with the Claude API*).

In [11]:
import html
import re
from IPython.display import HTML, display


def show_chunks(chunks, title_prefix="Chunk", show_length=True):
    """Bloques legibles en el notebook (evita stdout línea a línea en el IDE)."""
    total = len(chunks)
    for i, chunk in enumerate(chunks, 1):
        meta = f" · {len(chunk)} chars" if show_length else ""
        display(
            HTML(
                f'<div style="margin:14px 0;padding:14px 18px;border-left:5px solid #0969da;'
                f'background:#f6f8fa;border-radius:8px;font-family:ui-monospace,monospace;'
                f'font-size:13px;line-height:1.45;white-space:pre-wrap;word-break:break-word;">'
                f'<strong style="font-family:system-ui,sans-serif;color:#1f2328;">'
                f'{html.escape(title_prefix)} {i} of {total}{meta}</strong>'
                f'<div style="margin-top:10px;color:#24292f;">{html.escape(chunk)}</div>'
                f"</div>"
            )
        )


sample_doc = """## Executive Summary
Anthropic builds reliable, interpretable, and steerable AI systems. Our flagship model family, Claude, helps teams analyze documents, write code, and automate complex workflows.

## Engineering Metrics
During Q3, the platform engineering team resolved 142 production bugs and reduced median API latency by 28% across all regions.

## Medical Research Notes
A seasonal stomach bug affected several participants in the observational cohort, causing mild delays in follow-up scheduling.
"""

## 1. Size-Based Chunking (con overlap)
Divide el texto en bloques de longitud fija con solapamiento opcional para no perder contexto en los bordes.

In [2]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)

        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )

    return chunks

show_chunks(chunk_by_char(sample_doc, chunk_size=140, chunk_overlap=25), "Char chunk")

--- Chunk 1 (140 chars) ---
## Executive Summary
Anthropic builds reliable, interpretable, and steerable AI systems. Our flagship model family, Claude, helps teams anal

--- Chunk 2 (140 chars) ---
 Claude, helps teams analyze documents, write code, and automate complex workflows.

## Engineering Metrics
During Q3, the platform engineer

--- Chunk 3 (140 chars) ---
Q3, the platform engineering team resolved 142 production bugs and reduced median API latency by 28% across all regions.

## Medical Researc

--- Chunk 4 (140 chars) ---
ions.

## Medical Research Notes
A seasonal stomach bug affected several participants in the observational cohort, causing mild delays in fo

--- Chunk 5 (45 chars) ---
causing mild delays in follow-up scheduling.




## 2. Structure-Based Chunking
Divide el texto aprovechando encabezados (`## `) o secciones naturales del documento.

In [3]:
def chunk_by_section(document_text):
    pattern = r"\n## " # agrupa los textos por patron de markdown... con pattern.
    return re.split(pattern, document_text.strip())

show_chunks(chunk_by_section(sample_doc), "Section", show_length=False)

--- Section 1 ---
## Executive Summary
Anthropic builds reliable, interpretable, and steerable AI systems. Our flagship model family, Claude, helps teams analyze documents, write code, and automate complex workflows.


--- Section 2 ---
Engineering Metrics
During Q3, the platform engineering team resolved 142 production bugs and reduced median API latency by 28% across all regions.


--- Section 3 ---
Medical Research Notes
A seasonal stomach bug affected several participants in the observational cohort, causing mild delays in follow-up scheduling.



## 3. Sentence-Based Chunking
Divide por oraciones usando expresiones regulares y agrupa con solapamiento opcional.

In [4]:
def chunk_by_sentence(text, max_sentences_per_chunk=2, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip()) #split the text into individual sentences.

    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))

        start_idx += max_sentences_per_chunk - overlap_sentences

        if start_idx < 0:
            start_idx = 0

    return chunks

show_chunks(
    chunk_by_sentence(sample_doc, max_sentences_per_chunk=2, overlap_sentences=1),
    "Sentence chunk",
)

--- Sentence Chunk 1 ---
## Executive Summary
Anthropic builds reliable, interpretable, and steerable AI systems. Our flagship model family, Claude, helps teams analyze documents, write code, and automate complex workflows.

--- Sentence Chunk 2 ---
Our flagship model family, Claude, helps teams analyze documents, write code, and automate complex workflows. ## Engineering Metrics
During Q3, the platform engineering team resolved 142 production bugs and reduced median API latency by 28% across all regions.

--- Sentence Chunk 3 ---
## Engineering Metrics
During Q3, the platform engineering team resolved 142 production bugs and reduced median API latency by 28% across all regions. ## Medical Research Notes
A seasonal stomach bug affected several participants in the observational cohort, causing mild delays in follow-up scheduling.

--- Sentence Chunk 4 ---
## Medical Research Notes
A seasonal stomach bug affected several participants in the observational cohort, causing mild delays in fo

In [8]:
# esos ddefault settings no dan tan burenos resultados en la function de char

with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_char(text)
show_chunks(chunks, "report.md · char (defaults)")

# **Annual Interdisciplinary Research Review: Cross-Domain Insights**

## Executive Summary

This report synthesizes the key findings and ongoing rese
----

ngs and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the 
----

trength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disc
----

end traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** fo
----

edical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent st
----

ackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`). **Financial
----

7000E`). **Financial Analysis** revealed mixed quarterly p

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [12]:
# mejores default settings burenos resultados en la function de char

with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_char(text, 500, 150)
show_chunks(chunks, "report.md · char (500/150)")

In [13]:
# mejores default settings burenos resultados en la function de char

with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_sentence(text)
show_chunks(chunks, "report.md · by sentence")

In [14]:
# mejores default settings burenos resultados en la function de char

with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)
show_chunks(chunks, "report.md · by section")